### Checking how many files we get from the data (1000 items/file)

In [1]:
import os

# Define your path
path = 'crawled_data'
len_files = 0

# Check if the directory exists first to avoid errors
if os.path.exists(path):
    # List everything in the folder and filter for files only
    files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
    len_files = len(files)
    print(f"Total items in '{path}': {len_files}")
else:
    print(f"Directory '{path}' not found.")

Total items in 'crawled_data': 200


### Print out some of the product's description from different batch

In [2]:
import pandas as pd
import numpy as np

for i in range(5):
    rand_batch = np.random.randint(1,len_files + 1)
    df = pd.read_json(f"crawled_data/batch_{rand_batch}.json")
    rand_item = np.random.randint(len(df))
    print(f"Batch {rand_batch}, item {rand_item}")
    print(df["description"][rand_item])
    print("-------------------------------------------------------------------------------------------------")

Batch 171, item 91
<html>
 <head></head>
 <body>
  <ul>
   <p> Toà nhà N3A khu đô thị THNC</p>
  </ul>
 </body>
</html><p>Giá sản phẩm trên Tiki đã bao gồm thuế theo luật hiện hành. Bên cạnh đó, tuỳ vào loại sản phẩm, hình thức và địa chỉ giao hàng mà có thể phát sinh thêm chi phí khác như phí vận chuyển, phụ phí hàng cồng kềnh, thuế nhập khẩu (đối với đơn hàng giao từ nước ngoài có giá trị trên 1 triệu đồng).....</p>
-------------------------------------------------------------------------------------------------
Batch 173, item 349
<p>THẮT LƯNG NỮ FINE, DÂY NỊT NỮ<br />Thắt lưng nữ, chất liệu cao cấp, kiểu dáng thời trang hiện đại, là món phụ kiện tạo điểm nhấn cho bạn gái. Mua ngay thắt lưng nữ<br />Dây nịt nữ đẹp , That lung nữ hợp xu hướng thời trang Phong cách công sở, dự tiệc sang trọng<br />Thắt lưng váy đầm dựa trên kiểu váy đầm để lựa chọn mẫu thắt lưng thích hợp theo vóc dáng cơ thể sẽ giúp các nàng nổi bật hơn tạo vóc dáng thêm nữ tính và quyến rũ. 

### Now we create a function to apply masking on the "description" column to remove all HTML tags

In [3]:
import re
import html

def data_cleaner(text):
    if not text or pd.isna(text):
        return ""

    # 1. Handle Structural Tags
    # We replace these with newlines so the text stays in "paragraph" form
    # This covers p, br, div, li, tr, h1-h6, etc.
    text = re.sub(r'<(p|br|div|li|tr|h[1-6]|article|blockquote|ul|ol)[^>]*>', '\n', text)

    # 2. Strip all remaining tags (span, strong, html, body, head, etc.)
    # This removes anything else inside <>
    text = re.sub(r'<[^>]+>', '', text)

    # 3. Decode HTML entities
    # Turns &nbsp; into spaces, &amp; into &, etc.
    text = html.unescape(text)

    # 4. Clean up the "Invisible" characters
    # Tiki uses \xa0 (non-breaking space) frequently
    text = text.replace('\xa0', ' ').replace('\u200b', '')

    # 5. Spacing cleanup
    # Remove multiple spaces
    text = re.sub(r'[ \t]+', ' ', text)
    # Collapse 3+ newlines into 2 to keep it readable but compact
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)

    return text.strip()

In [4]:
from tqdm import tqdm
for batch in tqdm(range(1, len_files + 1), desc="Trimming HTML tags"):
    file_path = f"crawled_data/batch_{batch}.json"
    if os.path.exists(file_path):
        df = pd.read_json(file_path)
        df['description'] = df['description'].apply(data_cleaner)
        # Ensure it saves with Vietnamese characters intact
        df.to_json(file_path, orient='records', force_ascii=False, indent=4)

Trimming HTML tags: 100%|██████████| 200/200 [00:43<00:00,  4.55it/s]


### Get random item from random batch again to see the result

In [5]:
for i in range(5):
    rand_batch = np.random.randint(1, len_files + 1)
    df = pd.read_json(f"crawled_data/batch_{rand_batch}.json")
    rand_item = np.random.randint(len(df))
    print(f"Batch {rand_batch}, item {rand_item}")
    print(df["description"][rand_item])
    print("-------------------------------------------------------------------------------------------------")

Batch 117, item 473
1. Thông tin sản phẩm: Chân Váy Xếp Ly Dáng Midi Chất Tuytsi Cao Cấp Phong Cách Thanh Lịch

➤Thương hiệu: CChat Clothes

➤Màu sắc: Đen, Nâu, Ghi

➤Chất liệu: Tuytsi 

➤Sản phẩm bao gồm: 1 Chân váy 

➤Chân váy nữ phù hợp phong cách thanh lịch, nữ tính.

➤Mô tả chung: Vải tuytsi chất liệu mỏng, co giãn nhẹ, mặt nóng vuông, chân váy dập ly xòe quạt, thiết kế tạo eo ôm sát, tôn vòng eo thon, kết hợp với áo thun, áo kiểu, phù hợp với nhiều người.

➤Thông số size: S, M, L, XL 

2. Đặc điểm sản phẩm: Chân Váy Xếp Ly Dáng Midi Chất Tuytsi Cao Cấp Phong Cách Thanh Lịch

➤Chân váy được yêu thích bởi chất Tuytsi cao cấp, dày dặn, bề mặt có vân chéo rõ ràng, nhẹ và có độ thấm hút rất cao, mềm mại tạo cảm giác thoải mái khi mặc.

➤Tăng vẻ dịu dàng nữ tính, bởi thiết kế dập ly thủ công vô cùng tỉ mỉ từ CChat, giúp che khuyết điểm phần chân chưa hoàn hảo cho người mặc.

➤Tính ứng dụng: Chân váy nữ phù hợp mặc đi làm, đi chơi, đi hẹn hò dạo phố.

3. Hướng dẫn bảo quản sản phẩm

➤Lộ

In [6]:
import glob

# 1. Setup paths
input_dir = 'crawled_data'
output_file = 'tiki_products_final.csv'

# 2. Get a list of all JSON files
json_files = glob.glob(os.path.join(input_dir, 'batch_*.json'))
# Sort them so they are in order (optional)
json_files.sort(key=lambda x: int(os.path.basename(x).split('_')[1].split('.')[0]))

print(f"Found {len(json_files)} files to merge.")

# 3. Read and combine
all_data = []

for file in tqdm(json_files, desc="Merging Batches"):
    # Read individual JSON batch
    df_batch = pd.read_json(file)
    all_data.append(df_batch)

# 4. Concatenate all dataframes into one
final_df = pd.concat(all_data, ignore_index=True)

# 5. Save to CSV
# Use index=False so you don't get an extra 'Unnamed: 0' column
final_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\nDone! Saved {len(final_df)} rows to {output_file}")

Found 200 files to merge.


Merging Batches: 100%|██████████| 200/200 [00:09<00:00, 20.85it/s]



Done! Saved 180020 rows to tiki_products_final.csv
